# Уникальные скороговорки датасета

Подход:
1. Транскрибируем **все** WAV-файлы через Whisper.
2. Исправляем ошибки транскрипции через LLM (Ollama).
3. Сохраняем все записи с текстом и контрольными буквами в `data/tongue_twisters.csv`.
4. Считаем статистику уникальных скороговорок.

In [ ]:
import sys
import re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
from shared import config
from shared.data_utils import load_audio

CONTROL_LETTERS = list('лрстцчшщ')
print(f'DATA_DIR: {config.DATA_DIR}')

## 1. Загрузка Whisper

In [ ]:
WHISPER_ID = "openai/whisper-small"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Загружаем {WHISPER_ID} на {device}...")

processor = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper   = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID).to(device)
whisper.eval()
print("Готово.")

## 2. Вспомогательные функции

In [ ]:
import unicodedata

RUSSIAN_LOWER = frozenset(chr(c) for c in list(range(0x0430, 0x0450)) + [0x0451])

def fix_utf8_corrupted_on_windows(s):
    out = []
    i = 0
    while i < len(s):
        if i + 1 >= len(s):
            out.append(s[i]); i += 1; continue
        ord1, ord2 = ord(s[i]), ord(s[i+1])
        if ord2 < 0x80 or ord2 > 0xBF:
            if ord1 == 0x0430 and ord2 == 0x041B:
                byte2 = 0xBB
            else:
                out.append(s[i]); i += 1; continue
        else:
            byte2 = ord2
        first_byte = 0xD1 if ord1 == 0x0431 else (0xD0 if ord1 == 0x0430 else None)
        if first_byte is not None:
            try:
                out.append(bytes([first_byte, byte2]).decode('utf-8')); i += 2; continue
            except UnicodeDecodeError:
                pass
        out.append(s[i]); i += 1
    return ''.join(out)

def get_letter_combo(path):
    stem = Path(path).stem
    raw  = stem.split('__', 1)[1] if '__' in stem else ''
    decoded = fix_utf8_corrupted_on_windows(raw)
    return frozenset(c for c in decoded if c in RUSSIAN_LOWER)

def transcribe(path: str) -> str:
    y, _ = load_audio(path, sr=16000)
    inputs = processor(y, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)
    with torch.no_grad():
        ids = whisper.generate(
            input_features,
            language="russian",
            task="transcribe",
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

## 3. Транскрипция всех файлов (Whisper)

In [ ]:
paths_good = sorted(config.GOOD_DIR.glob('*.wav'))
paths_bad  = sorted(config.BAD_DIR.glob('*.wav'))
all_paths  = [(str(p), config.CLASS_GOOD) for p in paths_good] +              [(str(p), config.CLASS_BAD)  for p in paths_bad]

print(f"Всего файлов: {len(all_paths)}")

records = []
errors_whisper = []
for i, (path, label) in enumerate(all_paths):
    try:
        whisper_text = transcribe(path)
    except Exception as e:
        whisper_text = ''
        errors_whisper.append(f"{Path(path).name}: {e}")
    letters = get_letter_combo(path)
    row = {
        'filename':     Path(path).name,
        'label':        label,
        'whisper_text': whisper_text,
        'corrected_text': '',          # заполнится на следующем шаге
    }
    for letter in CONTROL_LETTERS:
        row[letter] = int(letter in letters)
    records.append(row)
    if (i + 1) % 100 == 0 or (i + 1) == len(all_paths):
        print(f"  {i+1}/{len(all_paths)}")

if errors_whisper:
    print(f"\nОШИБКИ ({len(errors_whisper)}):")
    for e in errors_whisper[:10]:
        print(f"  {e}")

df = pd.DataFrame(records)
print(f"\nПустых транскрипций: {(df.whisper_text == '').sum()}")
df.head(3)

## 4. Коррекция через Perplexity API

In [ ]:
import time
from openai import OpenAI

import os
API_KEY = os.environ["PERPLEXITY_API_KEY"]
pplx_client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.perplexity.ai",
)

SYSTEM_PROMPT = (
    "Ты помогаешь восстанавливать русские скороговорки по искажённой записи. "
    "Отвечай только одной строкой — каноническим вариантом скороговорки без перевода и комментариев. "
    "Если не уверен или такой скороговорки не существует, напиши ровно: НЕ УВЕРЕН."
)

def correct_text(raw: str) -> str:
    raw = raw.strip()
    if not raw:
        return ''
    resp = pplx_client.chat.completions.create(
        model="sonar",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": raw},
        ],
        temperature=0.1,
        max_tokens=128,
    )
    return resp.choices[0].message.content.strip()

print("Perplexity API клиент готов.")
print(f"Файлов для коррекции: {len(df)}")

In [ ]:
corrected = []
errors_llm = 0
for i, row in df.iterrows():
    try:
        text = correct_text(row['whisper_text'])
    except Exception as e:
        text = row['whisper_text']
        errors_llm += 1
        print(f"  Ошибка [{i}]: {e}")
    corrected.append(text)
    time.sleep(0.3)   # пауза чтобы не упереться в rate limit
    if (i + 1) % 100 == 0 or (i + 1) == len(df):
        print(f"  {i+1}/{len(df)}  (ошибок: {errors_llm})")

df['corrected_text'] = corrected
print(f"\nОшибок API: {errors_llm}")
df[['whisper_text', 'corrected_text']].head(5)

## 5. Сохранение

In [ ]:
col_order = ['filename', 'whisper_text', 'corrected_text'] + CONTROL_LETTERS
df_out = df[col_order].copy()

out_path = config.DATA_DIR / 'tongue_twisters.csv'
df_out.to_csv(out_path, index=False, encoding='utf-8')
print(f"Сохранено: {out_path}")
print(f"Записей: {len(df_out)}")
df_out.head(5)

## 6. Статистика

In [ ]:
import matplotlib.pyplot as plt

n_total       = len(df_out)
n_empty       = (df_out['whisper_text'] == '').sum()
n_unique_raw  = df_out[df_out['whisper_text'] != '']['whisper_text'].str.lower().str.strip().nunique()
n_unique_corr = df_out[df_out['corrected_text'] != '']['corrected_text'].str.lower().str.strip().nunique()

print("=" * 50)
print(f"Всего записей:            {n_total}")
print(f"Пустых (Whisper):         {n_empty}")
print(f"Уникальных (Whisper):     {n_unique_raw}")
print(f"Уникальных (после LLM):   {n_unique_corr}")
print("=" * 50)

# Топ-30 по частоте (по corrected_text)
top = (df_out[df_out['corrected_text'] != '']
       .groupby(df_out['corrected_text'].str.lower().str.strip())
       .agg(n=('filename', 'count'))
       .reset_index()
       .rename(columns={'corrected_text': 'text'})
       .sort_values('n', ascending=False)
       .head(30))

print(f"\nТоп-30 скороговорок:")
display(top.reset_index(drop=True))

counts_all = (df_out[df_out['corrected_text'] != '']
              .groupby(df_out['corrected_text'].str.lower().str.strip()).size()
              .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(top)), top['n'], color='steelblue')
axes[0].set_xticks(range(len(top)))
axes[0].set_xticklabels([t[:25] for t in top['text']], rotation=90, fontsize=6)
axes[0].set_ylabel('Количество записей')
axes[0].set_title('Топ-30 скороговорок')

axes[1].hist(counts_all.values, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Записей на скороговорку')
axes[1].set_ylabel('Количество скороговорок')
axes[1].set_title(f'Распределение ({n_unique_corr} уникальных после LLM)')
axes[1].axvline(counts_all.mean(), color='red', linestyle='--',
                label=f'среднее {counts_all.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(exp_dir / 'tongue_twisters_stats.png', dpi=150, bbox_inches='tight')
plt.show()